<a href="https://colab.research.google.com/github/dogyunyim/AI-NLP-TIL/blob/main/Subword_problem_(2026.06.25%20%EC%8B%A4%EC%8A%B5%EA%B3%BC%EC%A0%9C).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 서브 워드 비교 실험

서브워드 라이브러리 HuggingFace Tokenizer와 SentencePiece를 비교하는 과제 노트북입니다.

진행 순서는 다음과 같습니다.

1. 데이터 준비 : 네이버 영화 리뷰 [[링크]](https://raw.githubusercontent.com/e9t/nsmc/master/ratings_train.txt)
2. Sentencepiece/HuggingFace를 사용하여 단어사전 구축 및 속도 측정
3. 1과 2 결과물의 상위 빈도 30개의 서브워드 분석
4. 학습 속도(단어사전 구축 속도), 단일 문장 변환 속도, 코퍼스 변환 속도 차이 특정
5. 기타 추가 실험
6. 사용성과 장단점 분석

## 1. 데이터 준비 : 네이버 영화 리뷰

### HuggingFace Tokenizer와 SentencePiece 설치하기

In [1]:
!pip install -q sentencepiece tokenizers

### 네이버 리뷰 데이터 불러오기

In [2]:
import requests

url = "https://raw.githubusercontent.com/e9t/nsmc/master/ratings_train.txt"
response = requests.get(url)

with open("corpus.txt", "wb") as f:
    f.write(response.content)

In [3]:
import os

file_path = "/content/corpus.txt"

if os.path.exists(file_path):
    print("다운로드 완료")
else:
    print("파일을 찾을 수 없음")

다운로드 완료


### 데이터 정제하기
현재 데이터에는 `id`, `document`, `label`이 있습니다. 이 중 실질적인 문장 데이터인 `document`만 활용합니다.

In [4]:
# [[YOUR CODE]]
import pandas as pd

# 데이터 읽기
df = pd.read_csv("corpus.txt", sep="\t")

# document 컬럼만 추출
documents = df["document"]

# 비어있는 문장 제거
documents = documents.dropna()

# 새로운 파일 저장
with open("reviews.txt", "w", encoding="utf-8") as f:
    for doc in documents:
        f.write(doc + "\n")

print("정제 완료")
print(f"문장 개수: {len(documents)}")

정제 완료
문장 개수: 149995


## 2. Sentencepiece/HuggingFace를 사용하여 단어사전 구축 및 속도 측정

HuggingFace BPE는 SentencePiece와 내제된 철학이 다릅니다.  
고전적인 BPE는 단어 단위 알고리즘에서 출발하였으며 단어 목록을 입력으로 받아 문장 분리 -> 단어 분리 -> BPE를 가정합니다.  
때문에 이 철학을 따르는 HuggingFace BPE는 `Witespace()` 혹은 `ByteLevel()`과 같은 pre-tokenizer가 필요합니다.  
반면 SentencePiece는 공백도 학습하자는 철학으로 공백에 대하여 \_\_로 처리하여 기본 문장에서 공백을 기준으로 나누는 것은 동일하지만 문장 전체를 입력받게 되며 다음과 같이 처리합니다.  

"오늘은 날이 참 밝다."  
"오늘은\_\_날이\_\_참\_\_밝다."

만일 SentencePiece의 철학과 유사하게 진행하려면 `Metqaspace()`를 활용하십시오.

### Sentencepiece를 사용하여 단어사전 구축

In [5]:
# [[YOUR CODE]]
import sentencepiece as spm

spm.SentencePieceTrainer.train(
    input='reviews.txt',
    model_prefix='spm',
    vocab_size=5000,
    model_type='unigram'
)

### HuggingFace Tokenizer를 사용하여 단어사전 구축

In [6]:
# [[YOUR CODE]]
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace

# BPE 모델 생성
tokenizer = Tokenizer(BPE())

# 공백 기준으로 사전 분리
tokenizer.pre_tokenizer = Whitespace()

# 단어사전 설정
trainer = BpeTrainer(
    vocab_size=5000,
    special_tokens=["[UNK]"]
)

# 학습
tokenizer.train(["reviews.txt"], trainer)

# 저장
tokenizer.save("bpe_tokenizer.json")

print("BPE 학습 완료")

BPE 학습 완료


## 3. 1과 2 결과물의 상위 빈도 30개의 서브워드 분석

### 상위 30개 기준, 단어사전 조회하기

In [7]:
# [[YOUR CODE] setencepiece]
print("=== SentencePiece 상위 30개 ===")

with open("spm.vocab", "r", encoding="utf-8") as f:
    vocab = [line.strip().split("\t")[0] for line in f]

for i, token in enumerate(vocab[:30], 1):
    print(f"{i}. {token}")

=== SentencePiece 상위 30개 ===
1. <unk>
2. <s>
3. </s>
4. ▁
5. .
6. 이
7. ..
8. 가
9. ▁영화
10. 의
11. 도
12. ...
13. 는
14. 을
15. 고
16. 다
17. 에
18. ,
19. 은
20. 지
21. 한
22. ?
23. 만
24. 를
25. 로
26. 게
27. 나
28. 영화
29. ▁너무
30. !


In [8]:
# [[YOUR CODE] HuggingFace Tokenizer]
print("=== HuggingFace BPE 상위 30개 ===")

vocab = tokenizer.get_vocab()

sorted_vocab = sorted(
    vocab.items(),
    key=lambda x: x[1]
)

for i, (token, idx) in enumerate(sorted_vocab[:30], 1):
    print(f"{i}. {token}")

=== HuggingFace BPE 상위 30개 ===
1. [UNK]
2. !
3. "
4. #
5. $
6. %
7. &
8. '
9. (
10. )
11. *
12. +
13. ,
14. -
15. .
16. /
17. 0
18. 1
19. 2
20. 3
21. 4
22. 5
23. 6
24. 7
25. 8
26. 9
27. :
28. ;
29. <
30. =


### 분절된 토큰 빈도 조회

In [9]:
# [[YOUR CODE] SentencePiece ]
import sentencepiece as spm
from collections import Counter

sp = spm.SentencePieceProcessor()
sp.load("spm.model")

counter = Counter()

with open("reviews.txt", "r", encoding="utf-8") as f:
    for line in f:
        tokens = sp.encode(line.strip(), out_type=str)
        counter.update(tokens)

print("=== SentencePiece 토큰 빈도 TOP 30 ===")

for token, freq in counter.most_common(30):
    print(f"{token}: {freq}")

=== SentencePiece 토큰 빈도 TOP 30 ===
▁: 126199
.: 70246
이: 39361
..: 31201
가: 28585
의: 27384
▁영화: 26659
...: 25682
도: 25557
는: 22667
을: 21280
에: 19671
고: 19393
,: 18190
은: 17968
다: 17707
지: 16350
한: 14620
?: 12259
만: 11604
로: 11560
를: 11266
▁이: 11140
게: 10337
▁너무: 9904
나: 9567
리: 9508
!: 9242
영화: 9031
▁정말: 8938


In [10]:
# [[YOUR CODE] HuggingFace BPE ]
from collections import Counter

counter = Counter()

with open("reviews.txt", "r", encoding="utf-8") as f:
    for line in f:
        tokens = tokenizer.encode(line.strip()).tokens
        counter.update(tokens)

print("=== HuggingFace BPE 토큰 빈도 TOP 30 ===")

for token, freq in counter.most_common(30):
    print(f"{token}: {freq}")

=== HuggingFace BPE 토큰 빈도 TOP 30 ===
.: 69924
이: 36812
영화: 30955
..: 30149
...: 23826
의: 22941
도: 20336
을: 20173
가: 20047
한: 18437
다: 18307
,: 18087
에: 17870
지: 16690
은: 15875
는: 14536
고: 14109
?: 11996
아: 11085
너무: 11058
나: 10738
로: 10632
만: 9971
정말: 9814
어: 9808
를: 9491
기: 9453
!: 9204
수: 9000
안: 8969


## 4. 학습 속도(단어사전 구축 속도), 단일 문장 변환 속도, 코퍼스 변환 속도 차이 특정

### 단일 문장 변환 속도 측정

In [14]:
# [[YOUR CODE]]
import sentencepiece as spm

sp = spm.SentencePieceProcessor()
sp.load("spm.model")

print("로드 완료")


로드 완료


In [19]:
# [[YOUR CODE] SentencePiece ]
import time

start = time.time()

sentence = "이 영화는 정말 재미있고 감동적이었다."

sp.encode(sentence, out_type=str)

end = time.time()

print("SentencePiece 단일 문장 속도:", end - start)

SentencePiece 단일 문장 속도: 0.0002086162567138672


In [12]:
!ls

bpe_tokenizer.json  corpus.txt	reviews.txt  sample_data  spm.model  spm.vocab


In [17]:
# [[YOUR CODE] HuggingFace BPE ]
import time

sentence = "이 영화는 정말 재미있고 감동적이었다."

start = time.time()

tokenizer.encode(sentence)

end = time.time()

print("BPE 단일 문장 속도:", end - start)

BPE 단일 문장 속도: 0.0003230571746826172


### 코퍼스 전체 변환 속도 측정

In [20]:
# [[YOUR CODE] ]
import time

with open("reviews.txt", "r", encoding="utf-8") as f:
    corpus = f.readlines()

# SentencePiece
start = time.time()

for line in corpus:
    sp.encode(line.strip(), out_type=str)

sp_time = time.time() - start

# BPE
start = time.time()

for line in corpus:
    tokenizer.encode(line.strip())

bpe_time = time.time() - start

print(f"SentencePiece: {sp_time:.4f} sec")
print(f"BPE: {bpe_time:.4f} sec")

SentencePiece: 3.8015 sec
BPE: 4.7131 sec


## 5. 기타 추가 실험 (OOV 없는 단어 실험)


In [21]:
# [[YOUR CODE]]
test_sentences = [
    "챗봇자동화시스템",
    "멀티모달인공지능",
    "초거대언어모델",
    "생성형AI서비스",
    "텍스트마이닝분석",
    "딥러닝기반추천시스템"
]

for sent in test_sentences:
    print(f"\n문장: {sent}")
    print(sp.encode(sent, out_type=str))



문장: 챗봇자동화시스템
['▁', '챗', '봇', '자', '동', '화', '시', '스', '템']

문장: 멀티모달인공지능
['▁멀', '티', '모', '달', '인', '공', '지', '능']

문장: 초거대언어모델
['▁초', '거', '대', '언', '어', '모', '델']

문장: 생성형AI서비스
['▁생', '성', '형', 'A', 'I', '서', '비', '스']

문장: 텍스트마이닝분석
['▁', '텍', '스트', '마', '이', '닝', '분', '석']

문장: 딥러닝기반추천시스템
['▁', '딥', '러', '닝', '기', '반', '추천', '시', '스', '템']


## 6. 사용성과 장단점 분석

**SentencePiece**

**장점**  
공백을 포함하여 학습하기 때문에 별도의 전처리 없이 사용할 수 있었다.

한국어 조사 및 단어 단위가 자연스럽게 학습되는 것을 확인할 수 있었다.

단일 문장 변환 속도(0.00021초)와 전체 코퍼스 변환 속도(3.80초) 모두 BPE보다 빠르게 측정되었다.

사용 방법이 비교적 간단하여 빠르게 적용할 수 있었다.

**단점**
토큰 앞에 ▁ 기호가 붙어 결과를 해석할 때 다소 불편했다.

내부적으로 어떤 기준으로 분절되는지 직관적으로 이해하기 어려웠다.

**HuggingFace BPE**

**장점**

BPE 병합 과정을 수업에서 배운 내용과 연결하여 이해하기 쉬웠다.

토큰화 과정을 단계적으로 확인할 수 있어 학습용으로 적합했다.

HuggingFace 라이브러리와 쉽게 연동할 수 있었다.

**단점**

Whitespace Pre-tokenizer 설정이 필요하여 SentencePiece보다 준비 과정이 많았다.

실험 결과 단일 문장(0.00032초), 전체 코퍼스(4.71초) 모두 SentencePiece보다 느리게 측정되었다.

새로운 단어는 비교적 더 잘게 분절되는 경향을 보였다.